In [1]:
# ============================================
# LODO-валидация на новом сбалансированном датасете
# ============================================

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from utils.extract_features import extract_features

In [2]:

# Путь к новому датасету (сбалансированному)
NEW_DATASET_PATH = "../data/processed/balanced_unified_dataset.csv"  # укажите правильный путь!

# Загрузка
df_new = pd.read_csv(NEW_DATASET_PATH)
print(f"Загружено {len(df_new)} записей")
print(f"Классы: XSS={df_new[df_new.label==1].shape[0]}, Benign={df_new[df_new.label==0].shape[0]}")
print(f"Источники: {df_new['source'].unique()}")

# Определяем источники для теста (можно оставить все, но лучше исключить слишком маленькие)
source_counts = df_new['source'].value_counts()
sources_for_test = source_counts[source_counts > 1000].index.tolist()
print(f"\nИсточники для LODO (тест): {sources_for_test}")

# Параметры модели из вашего ModelConfig (можно использовать те же)
params = {
    'iterations': 10000,
    'depth': 6,
    'learning_rate': 0.05,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'early_stopping_rounds': 50,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'l2_leaf_reg': 3,
    'random_seed': 42,
    'task_type': 'CPU',
    'verbose': 100
}

def prepare_features(texts):
    features_list = []
    for text in texts:
        feat = extract_features(text)
        # Если extract_features возвращает словарь
        if isinstance(feat, dict):
            row = []
            for v in feat.values():
                try:
                    row.append(float(v))
                except (ValueError, TypeError):
                    row.append(0.0)
            features_list.append(row)
        # Если возвращает список или tuple
        elif isinstance(feat, (list, tuple)):
            row = []
            for v in feat:
                try:
                    row.append(float(v))
                except (ValueError, TypeError):
                    row.append(0.0)
            features_list.append(row)
        else:
            # Если вдруг вернулся одиночный объект
            try:
                features_list.append([float(feat)])
            except:
                features_list.append([0.0])
    # Проверка, что все строки одной длины (должны быть)
    if features_list:
        lengths = [len(row) for row in features_list]
        if len(set(lengths)) > 1:
            print(f"⚠️ Внимание: разные длины признаков! {set(lengths)}")
            # Дополним до максимальной длины нулями
            max_len = max(lengths)
            features_list = [row + [0.0]*(max_len - len(row)) for row in features_list]
    return pd.DataFrame(features_list)

lodo_results = []

for test_source in sources_for_test:
    print(f"\n{'='*60}")
    print(f"Тестовый источник: {test_source}")
    print('='*60)
    
    # Разделение
    train_df = df_new[df_new['source'] != test_source]
    test_df = df_new[df_new['source'] == test_source]
    print(f"Train: {len(train_df)} записей, Test: {len(test_df)} записей")
    
    # Извлечение признаков (используем ту же функцию extract_features)
    X_train = prepare_features(train_df['text'].tolist())
    X_test = prepare_features(test_df['text'].tolist())
    y_train = train_df['label'].values
    y_test = test_df['label'].values
    
    # Создание Pool
    train_pool = Pool(X_train, y_train)
    test_pool = Pool(X_test, y_test)
    
    # Обучение
    model = CatBoostClassifier(**params)
    model.fit(
        train_pool,
        eval_set=test_pool,
        use_best_model=True,
        verbose=100
    )
    
    # Предсказания
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    # Метрики
    pr_auc = average_precision_score(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    acc = np.mean(y_pred == y_test)
    
    # TPR@1%FPR
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    idx = np.argmin(np.abs(fpr - 0.01))
    tpr_at_1_fpr = tpr[idx] if idx < len(tpr) else 0.0
    
    lodo_results.append({
        'source': test_source,
        'n_train': len(train_df),
        'n_test': len(test_df),
        'pr_auc': pr_auc,
        'roc_auc': roc_auc,
        'f1': f1,
        'tpr@1%fpr': tpr_at_1_fpr,
        'accuracy': acc
    })
    
    print(f"PR-AUC: {pr_auc:.4f}, ROC-AUC: {roc_auc:.4f}, F1: {f1:.4f}, TPR@1%FPR: {tpr_at_1_fpr:.4f}, Acc: {acc:.4f}")

# Итоговая таблица
lodo_results_df = pd.DataFrame(lodo_results)
print("\n" + "="*60)
print("Результаты LODO по источникам")
print("="*60)
print(lodo_results_df.to_string(index=False))

# Средние значения
print("\n" + "="*60)
print("Средние метрики по всем источникам")
print("="*60)
means = lodo_results_df[['pr_auc', 'roc_auc', 'f1', 'tpr@1%fpr', 'accuracy']].mean()
for metric, value in means.items():
    print(f"{metric}: {value:.4f}")

# Сохраняем результаты
lodo_results_df.to_csv('./results/lodo_catboost_results.csv', index=False)
print("\nРезультаты сохранены в ./results/lodo_catboost_results.csv")

Загружено 70032 записей
Классы: XSS=35016, Benign=35016
Источники: ['fwaf' 'csic2010' 'vulnxss' 'kaggle' 'httpparams' 'payloadbox']

Источники для LODO (тест): ['fwaf', 'csic2010', 'kaggle', 'vulnxss', 'httpparams']

Тестовый источник: fwaf
Train: 35907 записей, Test: 34125 записей
0:	test: 0.0000000	best: 0.0000000 (0)	total: 131ms	remaining: 21m 49s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0
bestIteration = 0

Shrink model to first 1 iterations.
PR-AUC: 0.0000, ROC-AUC: nan, F1: 0.0000, TPR@1%FPR: nan, Acc: 0.7093

Тестовый источник: csic2010
Train: 44873 записей, Test: 25159 записей
0:	test: 0.0000000	best: 0.0000000 (0)	total: 13.3ms	remaining: 2m 12s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0
bestIteration = 0

Shrink model to first 1 iterations.
PR-AUC: 1.0000, ROC-AUC: nan, F1: 0.2230, TPR@1%FPR: 0.0000, Acc: 0.1255

Тестовый источник: kaggle
Train: 62689 записей, Test: 7343 записей
0:	test: 0.9910317	best: 0.9910317 (0)	total: 1